[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/MNPS_Likelihood_Evaluator_Severity.ipynb)

# MNPS Job Classification Likelihood Evaluator

## Purpose
This notebook evaluates how likely a given batch of model-generated job classifications is to match the classifications a **human evaluator** (with ~2 years of professional HR job classification experience) would make for the same roles.

---

## What this notebook does

For each job in a classification batch, this notebook:

1. **Loads real data** from:
   - `Sample JDs.csv` (original job descriptions)
   - `Job_Classifications_Batch.csv` (classifier output for the batch)
   - `Evaluation Resources.zip` containing:
     - `MNPS KSACs.csv`
     - `MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv`
     - `salary_by_major_role_grouping.csv`
     - `Ground Truth Masterfile.csv`
     - `Time to correct an error in hours.csv`

2. **Joins classifier output with ground truth** and role metadata

3. **Evaluates each record** using three components:
   - **KSAC similarity severity** (how close the predicted role is to the true role)
   - **Salary impact** (difference between typical pay for true vs predicted role)
   - **Time-to-correct impact** (how long it would take to fix a misclassification)

4. **Calibrates against human performance** expectations (accuracy range for a 2-year HR job classifier)

5. Produces a **0-5 Likelihood Score for each record**, where:
   - `5` = very likely that a human would classify the job the same way
   - `0` = extremely unlikely / severe mismatch

6. Saves **timestamped run results** to your Google Drive in:  
   `My Drive/Likelihood Assessment/Run Results/<YYYY-MM-DD_HH-MM-SS>/`

---

## High-level scoring idea

For each record, we compute a **severity index** based on:
- How similar the predicted role is to the true role (KSAC grouping)
- How big the salary gap is between the true and predicted role
- How much time an error like this would reasonably take to correct

Then we convert that severity into a **Likelihood Score (0-5)** and scale it by how the batch as a whole compares to a realistic **human baseline accuracy**.

## Features
- **Robust CSV handling**: Automatically handles all encodings and delimiters
- **Smart file discovery**: Checks multiple locations for input files
- **Comprehensive severity analysis**: KSAC + Salary + Time components
- **Human baseline calibration**: Scales scores against 2-year HR professional accuracy

In [ ]:
#============================================
# ENVIRONMENT SETUP AND IMPORTS
#============================================

import os
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

# Display options for easier debugging
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

print("✅ Environment setup complete")

In [ ]:
#============================================
# MOUNT GOOGLE DRIVE
#============================================

print("📂 Mounting Google Drive...")
drive.mount('/content/drive')

BASE_DRIVE_PATH = "/content/drive/My Drive"
BASE_RESULTS_ROOT = os.path.join(BASE_DRIVE_PATH, "Likelihood Assessment", "Run Results")
os.makedirs(BASE_RESULTS_ROOT, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"📁 Results root directory: {BASE_RESULTS_ROOT}")

In [ ]:
#============================================
# ROBUST CSV LOADING FUNCTION
#============================================

def load_csv_with_fallback(filepath, description="file"):
    """Load CSV with multiple encoding attempts and error handling"""
    
    print(f"\n📄 Loading {description}...")
    
    # Convert to Path object
    filepath = Path(filepath)
    
    if not filepath.exists():
        raise FileNotFoundError(f"❌ File not found: {filepath}")
    
    # List of encodings to try
    encodings = ['utf-8', 'latin1', 'iso-8859-1', 'cp1252', 'utf-16', 'windows-1252']
    
    # List of delimiters to try
    delimiters = [',', ';', '\t', '|']
    
    # Try each encoding
    for encoding in encodings:
        # Try each delimiter
        for delimiter in delimiters:
            try:
                if delimiter == ',':
                    print(f"   Trying {encoding} encoding...")
                else:
                    print(f"   Trying {encoding} encoding with '{delimiter}' delimiter...")
                
                df = pd.read_csv(filepath, encoding=encoding, sep=delimiter)
                
                # Verify we got actual data (at least 2 columns)
                if len(df.columns) >= 2:
                    if delimiter == ',':
                        print(f"   ✅ Successfully loaded with {encoding} encoding")
                    else:
                        print(f"   ✅ Successfully loaded with {encoding} encoding and '{delimiter}' delimiter")
                    print(f"   📊 Loaded {len(df)} rows with {len(df.columns)} columns")
                    return df
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue
            except Exception:
                continue
    
    # Last resort: try with error handling
    try:
        print(f"   Attempting with error handling...")
        df = pd.read_csv(filepath, encoding='utf-8', errors='ignore', on_bad_lines='skip')
        print(f"   ⚠️ Loaded with error handling (some characters may be lost)")
        print(f"   📊 Loaded {len(df)} rows with {len(df.columns)} columns")
        return df
    except Exception as e:
        raise Exception(f"❌ Failed to load {description} after trying all encodings: {e}")

print("✅ Robust CSV loading function initialized")
print("   Supports: UTF-8, Latin1, ISO-8859-1, CP1252, UTF-16, Windows-1252")
print("   Delimiters: comma, semicolon, tab, pipe")

In [ ]:
#============================================
# CONFIGURATION
#============================================

CONFIG = {
    # Where Evaluation Resources.zip lives
    "evaluation_zip_path": os.path.join(BASE_DRIVE_PATH, "Evaluation Resources.zip"),
    
    # Where to unzip the evaluation resources
    "evaluation_extract_dir": os.path.join(BASE_DRIVE_PATH, "Evaluation Resources"),
    
    # Names of the core resource files INSIDE the extracted folder
    "mnps_ksacs_filename": "MNPS KSACs.csv",
    "role_groups_filename": "MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv",
    "salary_by_role_filename": "salary_by_major_role_grouping.csv",
    "ground_truth_filename": "Ground Truth Masterfile.csv",
    "time_to_correct_filename": "Time to correct an error in hours.csv",
    
    # Batch input files
    "sample_jds_path": os.path.join(BASE_DRIVE_PATH, "Sample JDs.csv"),
    "batch_results_path": os.path.join(BASE_DRIVE_PATH, "Job_Classifications_Batch.csv"),
    
    # Column names used to join & compare
    "id_column": "source_row_index",
    "true_role_column": "expected_major_role_group",
    "pred_role_column": "major_role_group",
    
    # KSAC grouping columns
    "group_role_column": "major_role_group",
    "group_name_column": "similarity_group_id",
    
    # Salary-by-role columns
    "salary_role_column": "major_role_group",
    "salary_min_column": "min_salary",
    "salary_avg_column": "avg_salary",
    "salary_max_column": "max_salary",
    
    # Time-to-correct column
    "time_hours_column": "time_hours",
    
    # Human evaluator baseline accuracy
    "human_baseline_accuracy": 0.91,
    
    # Component weights for severity index
    "w_ksac": 0.5,
    "w_salary": 0.3,
    "w_time": 0.2,
}

print("✅ Configuration initialized")
print(f"\n📊 Component Weights:")
print(f"   KSAC Similarity: {CONFIG['w_ksac']*100}%")
print(f"   Salary Impact: {CONFIG['w_salary']*100}%")
print(f"   Time to Correct: {CONFIG['w_time']*100}%")
print(f"\n🎯 Human Baseline Accuracy: {CONFIG['human_baseline_accuracy']*100}%")

In [ ]:
#============================================
# LOAD EVALUATION RESOURCES
#============================================

def ensure_extracted(zip_path: str, extract_dir: str):
    """Extract evaluation resources if not already extracted"""
    if not os.path.exists(zip_path):
        # Try alternate locations
        alt_path = os.path.join("/content", "Evaluation Resources.zip")
        if os.path.exists(alt_path):
            zip_path = alt_path
        else:
            raise FileNotFoundError(f"❌ Evaluation zip not found at: {zip_path}")
    
    os.makedirs(extract_dir, exist_ok=True)
    
    # Only unzip if directory appears empty
    if not os.listdir(extract_dir):
        print(f"📦 Extracting {zip_path} -> {extract_dir}")
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_dir)
    else:
        print(f"✅ Using existing extracted resources at: {extract_dir}")

# Extract resources
ensure_extracted(CONFIG["evaluation_zip_path"], CONFIG["evaluation_extract_dir"])

# Load all evaluation resources
print("\n📂 Loading evaluation resources...")

eval_dir = Path(CONFIG["evaluation_extract_dir"])

# Find files (they might be in subdirectories)
def find_file(directory, filename):
    """Find file in directory or subdirectories"""
    for root, dirs, files in os.walk(directory):
        if filename in files:
            return os.path.join(root, filename)
    # Try case-insensitive match
    filename_lower = filename.lower()
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower() == filename_lower:
                return os.path.join(root, file)
    return None

# Load each resource file
mnps_ksacs_path = find_file(eval_dir, CONFIG["mnps_ksacs_filename"])
role_groups_path = find_file(eval_dir, CONFIG["role_groups_filename"])
salary_path = find_file(eval_dir, CONFIG["salary_by_role_filename"])
ground_truth_path = find_file(eval_dir, CONFIG["ground_truth_filename"])
time_path = find_file(eval_dir, CONFIG["time_to_correct_filename"])

mnps_ksacs_df = load_csv_with_fallback(mnps_ksacs_path, 'MNPS KSACs') if mnps_ksacs_path else pd.DataFrame()
role_groups_df = load_csv_with_fallback(role_groups_path, 'Role Groups') if role_groups_path else pd.DataFrame()
salary_df = load_csv_with_fallback(salary_path, 'Salary Data') if salary_path else pd.DataFrame()
ground_truth_df = load_csv_with_fallback(ground_truth_path, 'Ground Truth') if ground_truth_path else pd.DataFrame()
time_to_correct_df = load_csv_with_fallback(time_path, 'Time to Correct') if time_path else pd.DataFrame()

print("\n✅ All evaluation resources loaded!")

In [ ]:
#============================================
# LOAD SAMPLE JDs AND BATCH RESULTS
#============================================

print("\n📂 Loading input data files...")

# Check multiple locations
def find_input_file(filename):
    """Find input file in Drive or /content"""
    locations = [
        os.path.join(BASE_DRIVE_PATH, filename),
        os.path.join("/content", filename),
    ]
    for loc in locations:
        if os.path.exists(loc):
            return loc
    raise FileNotFoundError(f"❌ Could not find {filename} in Drive or /content")

sample_jds_path = find_input_file("Sample JDs.csv")
batch_results_path = find_input_file("Job_Classifications_Batch.csv")

sample_jds_df = load_csv_with_fallback(sample_jds_path, 'Sample JDs')
batch_results_df = load_csv_with_fallback(batch_results_path, 'Job Classifications Batch')

print("\n✅ All input files loaded successfully!")
print(f"   Sample JDs: {len(sample_jds_df)} rows")
print(f"   Batch Results: {len(batch_results_df)} rows")
print(f"   Ground Truth: {len(ground_truth_df)} rows")

In [ ]:
#============================================
# HELPER FUNCTIONS
#============================================

def normalize_role_name(role):
    """Basic normalization for role names"""
    if pd.isna(role):
        return None
    s = str(role).strip()
    if not s:
        return None
    return s

# Build role → KSAC group mapping
group_role_col = CONFIG["group_role_column"]
group_name_col = CONFIG["group_name_column"]

if not role_groups_df.empty and group_role_col in role_groups_df.columns and group_name_col in role_groups_df.columns:
    role_groups_df[group_role_col] = role_groups_df[group_role_col].apply(normalize_role_name)
    role_group_map = (
        role_groups_df
        .dropna(subset=[group_role_col, group_name_col])
        .drop_duplicates(subset=[group_role_col])
        .set_index(group_role_col)[group_name_col]
        .to_dict()
    )
    print(f"✅ Role→Group mapping entries: {len(role_group_map)}")
else:
    role_group_map = {}
    print("⚠️ No role group mapping available")

# Salary mapping
salary_role_col = CONFIG["salary_role_column"]

if not salary_df.empty and salary_role_col in salary_df.columns:
    salary_df[salary_role_col] = salary_df[salary_role_col].apply(normalize_role_name)
    
    min_col = CONFIG["salary_min_column"]
    avg_col = CONFIG["salary_avg_column"]
    max_col = CONFIG["salary_max_column"]
    
    # Check if columns exist, otherwise use first 3 numeric columns
    if not all(c in salary_df.columns for c in [min_col, avg_col, max_col]):
        numeric_cols = salary_df.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) >= 3:
            min_col, avg_col, max_col = numeric_cols[:3]
            print(f"⚠️ Using first three numeric columns: {min_col}, {avg_col}, {max_col}")
    
    if all(c in salary_df.columns for c in [min_col, avg_col, max_col]):
        salary_map = (
            salary_df
            .set_index(salary_role_col)[[min_col, avg_col, max_col]]
            .to_dict(orient="index")
        )
        
        global_min_salary = salary_df[min_col].min()
        global_max_salary = salary_df[max_col].max()
        salary_range = max(global_max_salary - global_min_salary, 1.0)
        
        print(f"✅ Salary range: ${global_min_salary:,.0f} - ${global_max_salary:,.0f}")
    else:
        salary_map = {}
        salary_range = 50000
        global_min_salary = 30000
        global_max_salary = 80000
        print("⚠️ Using default salary range")
else:
    salary_map = {}
    salary_range = 50000
    global_min_salary = 30000
    global_max_salary = 80000
    print("⚠️ No salary mapping available, using defaults")

# Time-to-correct mapping
if not time_to_correct_df.empty:
    hours_col = CONFIG["time_hours_column"]
    if hours_col not in time_to_correct_df.columns:
        numeric_cols = time_to_correct_df.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_cols:
            hours_col = numeric_cols[0]
            print(f"⚠️ Using first numeric column as hours: {hours_col}")
    
    if hours_col and hours_col in time_to_correct_df.columns:
        min_hours = float(time_to_correct_df[hours_col].min())
        max_hours = float(time_to_correct_df[hours_col].max())
        hours_range = max(max_hours - min_hours, 1.0)
        print(f"✅ Time range: {min_hours:.1f} - {max_hours:.1f} hours")
    else:
        min_hours = 0.5
        max_hours = 8.0
        hours_range = 7.5
        print("⚠️ Using default time range")
else:
    min_hours = 0.5
    max_hours = 8.0
    hours_range = 7.5
    print("⚠️ No time correction data available, using defaults")

def estimate_time_severity(expected_hours: float) -> float:
    """Normalize hours to [0,1] severity"""
    if expected_hours is None or np.isnan(expected_hours):
        return 0.5
    return (expected_hours - min_hours) / hours_range

def estimate_hours_from_severity(severity: float) -> float:
    """Map severity [0,1] back to hours"""
    severity = max(0.0, min(1.0, float(severity)))
    return min_hours + severity * hours_range

print("\n✅ All helper functions and mappings initialized")

In [ ]:
#============================================
# MERGE GROUND TRUTH AND PREDICTIONS
#============================================

print("\n🔄 Merging ground truth with predictions...")

id_col = CONFIG["id_column"]
true_role_col = CONFIG["true_role_column"]
pred_role_col = CONFIG["pred_role_column"]

# Normalize role names
if true_role_col in ground_truth_df.columns:
    ground_truth_df[true_role_col] = ground_truth_df[true_role_col].apply(normalize_role_name)

if pred_role_col in batch_results_df.columns:
    batch_results_df[pred_role_col] = batch_results_df[pred_role_col].apply(normalize_role_name)

# Merge
if id_col in ground_truth_df.columns and id_col in batch_results_df.columns:
    merged_df = ground_truth_df.merge(
        batch_results_df[[id_col, pred_role_col]],
        on=id_col,
        how="inner",
        suffixes=("_truth", "_pred")
    )
    print(f"✅ Merged {len(merged_df)} records")
else:
    raise ValueError(f"❌ ID column '{id_col}' not found in both datasets")

if merged_df.empty:
    raise ValueError("❌ Merged dataset is empty. Check ID column alignment.")

# Attach KSAC group info
merged_df["true_group"] = merged_df[true_role_col].map(role_group_map)
merged_df["pred_group"] = merged_df[pred_role_col].map(role_group_map)

# Attach salary info
def get_salary_dict(role_name):
    if role_name is None or role_name not in salary_map:
        return {"min": np.nan, "avg": np.nan, "max": np.nan}
    entry = salary_map[role_name]
    return {
        "min": entry.get(min_col, np.nan),
        "avg": entry.get(avg_col, np.nan),
        "max": entry.get(max_col, np.nan),
    }

salary_true = merged_df[true_role_col].apply(get_salary_dict)
salary_pred = merged_df[pred_role_col].apply(get_salary_dict)

merged_df["true_salary_min"] = [d["min"] for d in salary_true]
merged_df["true_salary_avg"] = [d["avg"] for d in salary_true]
merged_df["true_salary_max"] = [d["max"] for d in salary_true]

merged_df["pred_salary_min"] = [d["min"] for d in salary_pred]
merged_df["pred_salary_avg"] = [d["avg"] for d in salary_pred]
merged_df["pred_salary_max"] = [d["max"] for d in salary_pred]

print(f"✅ Added KSAC groups and salary information")
print(f"\n📊 Preview of merged data:")
display(merged_df.head(5))

In [ ]:
#============================================
# COMPUTE SEVERITY AND LIKELIHOOD SCORES
#============================================

print("\n🔢 Computing severity scores...")

w_ksac = CONFIG["w_ksac"]
w_salary = CONFIG["w_salary"]
w_time = CONFIG["w_time"]
human_baseline_accuracy = CONFIG["human_baseline_accuracy"]

def compute_ksac_severity(row) -> float:
    """Return severity score [0,1] based on KSAC similarity"""
    true_role = row[true_role_col]
    pred_role = row[pred_role_col]
    true_group = row["true_group"]
    pred_group = row["pred_group"]
    
    if pd.isna(pred_role):
        return 1.0  # No prediction = maximum severity
    if true_role == pred_role:
        return 0.0  # Exact match
    if pd.isna(true_group) or pd.isna(pred_group):
        return 0.6  # Missing group info = medium severity
    if true_group == pred_group:
        return 0.3  # Same group = low severity
    else:
        return 0.9  # Different group = high severity

def compute_salary_impact(row) -> float:
    """Return salary impact severity [0,1]"""
    true_avg = row["true_salary_avg"]
    pred_avg = row["pred_salary_avg"]
    
    if pd.isna(true_avg) or pd.isna(pred_avg):
        return 0.5  # Missing data = medium severity
    
    diff = abs(pred_avg - true_avg)
    return min(diff / salary_range, 1.0)

def compute_time_severity(row, ksac_severity: float, salary_impact: float) -> float:
    """Estimate time-to-correct severity"""
    combined = 0.6 * ksac_severity + 0.4 * salary_impact
    hours = estimate_hours_from_severity(combined)
    return estimate_time_severity(hours)

# Compute severities
merged_df["ksac_severity"] = merged_df.apply(compute_ksac_severity, axis=1)
merged_df["salary_impact"] = merged_df.apply(compute_salary_impact, axis=1)
merged_df["time_severity"] = merged_df.apply(
    lambda r: compute_time_severity(r, r["ksac_severity"], r["salary_impact"]), axis=1
)

# Overall severity index
total_weight = w_ksac + w_salary + w_time
merged_df["severity_index"] = (
    (w_ksac * merged_df["ksac_severity"] +
     w_salary * merged_df["salary_impact"] +
     w_time * merged_df["time_severity"]) / total_weight
)

# Batch-level accuracy
merged_df["is_exact_match"] = merged_df[true_role_col] == merged_df[pred_role_col]
classifier_accuracy = merged_df["is_exact_match"].mean() if len(merged_df) > 0 else 0.0

print(f"\n📊 Batch Statistics:")
print(f"   Classifier accuracy: {classifier_accuracy:.1%}")
print(f"   Human baseline: {human_baseline_accuracy:.1%}")

# Scale factor
global_scale = min(1.0, classifier_accuracy / max(human_baseline_accuracy, 1e-6))
print(f"   Global scale factor: {global_scale:.3f}")

# Convert to 0-5 likelihood score
merged_df["likelihood_base_0_to_5"] = 5.0 * (1.0 - merged_df["severity_index"].clip(0.0, 1.0))
merged_df["likelihood_score_0_to_5"] = (merged_df["likelihood_base_0_to_5"] * global_scale).clip(0.0, 5.0)

# Severity buckets
def bucket_severity(sev: float) -> str:
    if sev <= 0.1:
        return "No error / negligible"
    if sev <= 0.3:
        return "Low"
    if sev <= 0.6:
        return "Moderate"
    if sev <= 0.85:
        return "High"
    return "Severe"

merged_df["severity_bucket"] = merged_df["severity_index"].apply(bucket_severity)

print(f"\n✅ Severity and likelihood scores computed")
print(f"\n📊 Sample Results:")
display(
    merged_df[[
        id_col,
        true_role_col, "true_group",
        pred_role_col, "pred_group",
        "ksac_severity", "salary_impact", "time_severity",
        "severity_index", "likelihood_score_0_to_5"
    ]].head(10)
)

In [ ]:
#============================================
# BATCH SUMMARY AND DIAGNOSTICS
#============================================

print("\n" + "="*80)
print("BATCH EVALUATION SUMMARY")
print("="*80)

n = len(merged_df)
exact_acc = merged_df["is_exact_match"].mean()
avg_likelihood = merged_df["likelihood_score_0_to_5"].mean()

print(f"\n📊 Overall Statistics:")
print(f"   Total records evaluated: {n}")
print(f"   Exact-match accuracy: {exact_acc:.1%}")
print(f"   Average Likelihood Score: {avg_likelihood:.2f} / 5.0")

print(f"\n📊 Severity Distribution:")
bucket_counts = merged_df["severity_bucket"].value_counts().reindex(
    ["No error / negligible", "Low", "Moderate", "High", "Severe"],
    fill_value=0
)
for bucket, count in bucket_counts.items():
    pct = count / n if n > 0 else 0
    print(f"   {bucket:25s}: {count:5d}  ({pct:5.1%})")

print(f"\n⚠️ Top 5 Severe Cases:")
severe_examples = merged_df.sort_values("severity_index", ascending=False).head(5)
display(
    severe_examples[[
        id_col,
        true_role_col, pred_role_col,
        "severity_index", "likelihood_score_0_to_5", "severity_bucket"
    ]]
)

print("\n" + "="*80)

In [ ]:
#============================================
# SAVE RESULTS
#============================================

print("\n💾 Saving results...")

timestamp_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_dir = os.path.join(BASE_RESULTS_ROOT, timestamp_str)
os.makedirs(run_dir, exist_ok=True)

results_csv_path = os.path.join(run_dir, "likelihood_scored_results.csv")
summary_csv_path = os.path.join(run_dir, "likelihood_batch_summary.csv")

# Save full results
try:
    merged_df.to_csv(results_csv_path, index=False, encoding='utf-8')
except Exception:
    merged_df.to_csv(results_csv_path, index=False, encoding='latin1')

# Save summary
summary_data = {
    "timestamp": [timestamp_str],
    "records_evaluated": [len(merged_df)],
    "exact_match_accuracy": [exact_acc],
    "average_likelihood_score": [avg_likelihood],
    "human_baseline_accuracy": [human_baseline_accuracy],
    "global_scale_factor": [global_scale]
}
summary_df = pd.DataFrame(summary_data)

try:
    summary_df.to_csv(summary_csv_path, index=False, encoding='utf-8')
except Exception:
    summary_df.to_csv(summary_csv_path, index=False, encoding='latin1')

print(f"\n✅ Results saved successfully!")
print(f"📁 Run folder: {run_dir}")
print(f"   📄 Per-record results: likelihood_scored_results.csv")
print(f"   📄 Batch summary: likelihood_batch_summary.csv")
print(f"\n🕐 Timestamp: {timestamp_str}")

In [ ]:
#============================================
# FINAL SUMMARY
#============================================

print("\n" + "="*80)
print("✅ EVALUATION COMPLETE")
print("="*80)
print(f"\n📊 Final Results:")
print(f"   Records evaluated: {n}")
print(f"   Exact-match accuracy: {exact_acc:.1%}")
print(f"   Average Likelihood Score: {avg_likelihood:.2f} / 5.0")
print(f"   Human baseline: {human_baseline_accuracy:.1%}")
print(f"\n📁 Results location: {run_dir}")
print("\n" + "="*80)